# Sample Patches for Quality Assessment

This notebook samples random patches from annotation JSONs until the
cumulative cell count reaches ~3 000 cells (3 batches × ~1 000 cells each).
It then downloads the selected tile images from the workspace blob store and
compresses everything into a single `.zip`.

**Inputs required:**
- `ANNOTATIONS_ZIP` — path to `combined_pipeline_outputs.zip` on this compute instance
- `TILES_BASE_URIS` — list of `azureml://` URIs pointing to `tile_filter/` outputs (slides may be spread across multiple URIs)

**Outputs (per batch):**
- `qa_sample/batch_N/` — tile PNG images
- `qa_sample/batch_N/cell_labels.json` — full cell polygons + labels for those tiles
- `qa_sample/manifest.json` — sampling metadata
- `qa_sample.zip` — everything compressed for download

## 1. Configuration

In [29]:
# Path to the zip file containing per-slide annotation JSONs.
# Inside the zip: <ANNOTATIONS_SUBDIR>/<slide_id>.json
ANNOTATIONS_ZIP    = "combined_pipeline_outputs.zip"
ANNOTATIONS_SUBDIR = "pipeline_outputs_Feb_2026_cellpose_4_0_8_cpsam"

# AzureML datastore URIs for the tile_filter output.
# Slides may be spread across different datastores / paths.
# List all base URIs — each should contain sub-dirs per slide_id with .png patches.
TILES_BASE_URIS = [
    "azureml://subscriptions/bbeb7561-f822-4950-82f6-64dcae8a93ab/resourcegroups/AIMLCC-DEV-RG/workspaces/edu06_histology_img_segmentation/datastores/workspaceblobstore/paths/20260212_121112_v4_model_cpsam_prob_0pt0_flow_0pt4_eps_0pt5_mins_5_segGPU_norm_cluGPU_umap_50_perslide_mag_1_filtered_edge_0pt02_annotated_reptiles_0pt0_maxper_10_filteredAnnotated/tile_filter",
    "azureml://subscriptions/bbeb7561-f822-4950-82f6-64dcae8a93ab/resourcegroups/AIMLCC-DEV-RG/workspaces/edu06_histology_img_segmentation/datastores/workspaceblobstore/paths/20260121_140250_v4_model_cpsam_prob_0pt0_flow_0pt4_eps_0pt5_mins_5_segGPU_norm_cluGPU_umap_50_perslide_mag_1_filtered_edge_0pt02_annotated_reptiles_0pt0_maxper_10_filteredAnnotated/tile_filter/"
]

# Number of batches and target cells per batch
NUM_BATCHES     = 3
CELLS_PER_BATCH = 1000

# Random seed for reproducibility (set to None for non-reproducible)
RANDOM_SEED = 42

# Output directory & zip name
OUTPUT_DIR = "qa_sample"
ZIP_NAME   = "qa_sample.zip"

## 2. Set up AzureML datastore filesystems

`azureml-fsspec` provides an fsspec-compatible filesystem that speaks
`azureml://` URIs directly.  On an AzureML compute instance, the managed
identity is picked up automatically — no storage account name or SAS token
is needed.

We create one filesystem per base URI and verify connectivity on each.

In [30]:
# !pip install azureml-fsspec   # uncomment and run once if not already installed

from azureml.fsspec import AzureMachineLearningFileSystem

# Create one filesystem per base URI
fs_map = {}  # base_uri -> AzureMachineLearningFileSystem
for uri in TILES_BASE_URIS:
    fs_map[uri] = AzureMachineLearningFileSystem(uri)

print(f"Created {len(fs_map)} filesystem(s) for {len(TILES_BASE_URIS)} base URI(s).")

Created 2 filesystem(s) for 2 base URI(s).


In [ ]:
# Verify datastore connectivity and build slide_id -> base_uri mapping
slide_uri_map = {}  # slide_id -> base_uri

for uri, fs in fs_map.items():
    print(f"\nProbing: {uri}")
    try:
        entries = fs.ls(uri, detail=False)
        # Each entry is a slide sub-directory
        slide_ids_found = [e.rstrip("/").split("/")[-1] for e in entries]
        for sid in slide_ids_found:
            if sid not in slide_uri_map:
                slide_uri_map[sid] = uri
        print(f"  ✓ Accessible — {len(slide_ids_found)} slide sub-dir(s) found")
        if slide_ids_found[:3]:
            print(f"    First entries: {slide_ids_found[:3]}")
    except Exception as e:
        print(f"  ✗ Could not access – check the URI and permissions.\n    {e}")

print(f"\nTotal slides mapped across all URIs: {len(slide_uri_map)}")

## 3. Build per-slide tile index from annotation zip

Read each `<slide_id>.json` from the zip and build a dict keyed by slide ID.
Only non-empty tiles (with at least one cell) are kept. Cell polygons are
**not** stored — only the count — to keep memory low.

In [ ]:
import json, os, zipfile
from pathlib import Path, PurePosixPath

# Open the zip and discover annotation JSONs in the expected sub-directory
zf = zipfile.ZipFile(ANNOTATIONS_ZIP)
ann_entries = sorted(
    name for name in zf.namelist()
    if name.startswith(ANNOTATIONS_SUBDIR + "/") and name.endswith(".json")
)

print(f"Found {len(ann_entries)} annotation JSON(s) inside zip:")
for name in ann_entries:
    info = zf.getinfo(name)
    print(f"  {PurePosixPath(name).name}  ({info.file_size / 1e6:.1f} MB compressed)")

### Read annotation files from zip and build per-slide tile index

In [ ]:
slide_tiles = {}  # slide_id -> list of tile dicts
total_tiles = 0
total_cells = 0

for entry_name in ann_entries:
    slide_id = PurePosixPath(entry_name).stem
    print(f"Indexing {slide_id} …", end=" ", flush=True)

    with zf.open(entry_name) as f:
        tiles = json.load(f)

    slide_list = []
    for tile_ann in tiles:
        n_cells = len(tile_ann.get("cells", []))
        if n_cells == 0:
            continue
        slide_list.append({
            "slide_id":      slide_id,
            "tile_path":     tile_ann["tile_path"],
            "tile_name":     os.path.basename(tile_ann["tile_path"]),
            "cell_count":    n_cells,
            "magnification": tile_ann.get("magnification"),
            "x":             tile_ann.get("x"),
            "y":             tile_ann.get("y"),
        })

    slide_tiles[slide_id] = slide_list
    slide_cells = sum(t["cell_count"] for t in slide_list)
    total_tiles += len(slide_list)
    total_cells += slide_cells
    del tiles
    print(f"{len(slide_list)} non-empty tiles, {slide_cells:,} cells")

print(f"\nTotal tiles with cells : {total_tiles:,}")
print(f"Total cells across all : {total_cells:,}")

## 4. Random sampling — 3 batches of ~1 000 cells each

Each slide gets an equal cell budget (`total_target / num_slides`).
Tiles within each slide are shuffled and greedily picked until the
budget is reached, then all selected tiles are shuffled and split
into batches.

In [ ]:
import random

rng = random.Random(RANDOM_SEED)
total_target = NUM_BATCHES * CELLS_PER_BATCH
num_slides = len(slide_tiles)
per_slide_budget = total_target / num_slides
print(f"Target: ~{total_target:,} cells from {num_slides} slides → ~{per_slide_budget:.0f} cells/slide\n")

# ---- 1. Per-slide greedy sampling up to budget ----
all_sampled = []
for slide_id in sorted(slide_tiles):
    tiles = slide_tiles[slide_id].copy()
    rng.shuffle(tiles)
    picked, cells = [], 0
    for t in tiles:
        picked.append(t)
        cells += t["cell_count"]
        if cells >= per_slide_budget:
            break
    all_sampled.extend(picked)
    print(f"  {slide_id}: {len(picked)} tiles, {cells:,} cells")

# ---- 2. Shuffle and assign to batches ----
rng.shuffle(all_sampled)

batches = []
remaining = iter(all_sampled)
for batch_idx in range(1, NUM_BATCHES + 1):
    batch_tiles = []
    batch_cells = 0
    for tile in remaining:
        tile["batch"] = batch_idx
        batch_tiles.append(tile)
        batch_cells += tile["cell_count"]
        if batch_cells >= CELLS_PER_BATCH:
            break
    batches.append(batch_tiles)
    print(f"\nBatch {batch_idx}: {len(batch_tiles)} tiles, {batch_cells:,} cells")

sampled_tiles = [t for b in batches for t in b]
cumulative_cells = sum(t["cell_count"] for t in sampled_tiles)
print(f"\nTotal: {len(sampled_tiles)} tiles  |  {cumulative_cells:,} cells across {NUM_BATCHES} batches")

## 6. Download the sampled tile images from the datastore

Each tile's `tile_path` encodes the slide ID and patch filename. We look up
the correct base URI for each slide via `slide_uri_map` (built in Step 2)
and download from the corresponding filesystem.

In [ ]:
def resolve_tile(tile_path: str, slide_id: str):
    """Return (azureml_uri, filesystem) for a tile using slide_uri_map.

    tile_path looks like:
        /mnt/azureml/.../INPUT_prepped_tiles_path/<slide_id>/<patch>.png

    We extract <slide_id>/<patch>.png and prepend the base URI for that slide.
    """
    parts = tile_path.replace("\\", "/").split("/")
    try:
        idx = next(i for i, p in enumerate(parts) if p.startswith("INPUT_prepped_tiles_path"))
        rel = "/".join(parts[idx + 1:])   # slide_id/patch.png
    except StopIteration:
        rel = "/".join(parts[-2:])

    base_uri = slide_uri_map.get(slide_id)
    if base_uri is None:
        raise KeyError(f"Slide '{slide_id}' not found in any base URI. "
                       f"Check TILES_BASE_URIS or datastore contents.")

    uri = f"{base_uri.rstrip('/')}/{rel}"
    return uri, fs_map[base_uri]

# Quick sanity check
sample = sampled_tiles[0]
uri, _ = resolve_tile(sample["tile_path"], sample["slide_id"])
print(f"Recorded tile_path : {sample['tile_path']}")
print(f"Resolved URI       : {uri}")
print(f"Base URI used      : {slide_uri_map[sample['slide_id']]}")

In [ ]:
import shutil

# Clean output directory and create batch subdirs
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
for batch_idx in range(1, NUM_BATCHES + 1):
    os.makedirs(os.path.join(OUTPUT_DIR, f"batch_{batch_idx}"), exist_ok=True)

print(f"Downloading {len(sampled_tiles)} tile images into {NUM_BATCHES} batch dirs...")
failed = []

for i, tile in enumerate(sampled_tiles):
    try:
        uri, fs = resolve_tile(tile["tile_path"], tile["slide_id"])
    except KeyError as e:
        failed.append((tile["tile_path"], str(e)))
        tile["local_path"] = None
        continue

    local_name = f"{tile['slide_id']}__{tile['tile_name']}"
    if not local_name.endswith(".png"):
        local_name += ".png"
    local_path = os.path.join(OUTPUT_DIR, f"batch_{tile['batch']}", local_name)

    try:
        with fs.open(uri, "rb") as src, open(local_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
        tile["local_path"] = local_path
    except Exception as e:
        failed.append((uri, str(e)))
        tile["local_path"] = None

    if (i + 1) % 20 == 0 or (i + 1) == len(sampled_tiles):
        print(f"  [{i+1}/{len(sampled_tiles)}] downloaded")

if failed:
    print(f"\n⚠ {len(failed)} tiles failed to download:")
    for ref, err in failed[:5]:
        print(f"  {ref}: {err}")
    if len(failed) > 5:
        print(f"  ... and {len(failed) - 5} more")
else:
    print("\nAll tiles downloaded successfully.")

## 7. Extract cell labels for sampled tiles

Re-read the annotation JSONs from the zip, extracting full cell data only
for the tiles selected in Step 5.  One `cell_labels.json` is written per
batch directory.

The output schema is:
```json
{
  "batch": 1,
  "tiles": [
    {
      "slide_id": "...", "tile_name": "...", "tile_path": "...",
      "magnification": 1.0, "x": 2048, "y": 1024,
      "cells": [ {"cell_id": 1, "polygon": [...], "pred_class": "...", ...} ]
    }
  ]
}
```

In [ ]:
from collections import defaultdict

# Fast lookups
tile_lookup      = {(t["slide_id"], t["tile_path"]): t for t in sampled_tiles}
slide_to_wanted  = defaultdict(set)
for (slide_id, tile_path) in tile_lookup:
    slide_to_wanted[slide_id].add(tile_path)

batch_tile_data = {b: [] for b in range(1, NUM_BATCHES + 1)}

for entry_name in ann_entries:
    slide_id = PurePosixPath(entry_name).stem
    if slide_id not in slide_to_wanted:
        continue  # no sampled tiles from this slide – skip

    wanted_paths = slide_to_wanted[slide_id]
    print(f"Scanning {slide_id} for {len(wanted_paths)} tile(s)…", end=" ", flush=True)

    with zf.open(entry_name) as f:
        tiles = json.load(f)

    found = 0
    for tile_ann in tiles:
        tile_path = tile_ann.get("tile_path", "")
        if tile_path not in wanted_paths:
            continue

        meta = tile_lookup[(slide_id, tile_path)]
        batch_tile_data[meta["batch"]].append({
            "slide_id":      slide_id,
            "tile_name":     meta["tile_name"],
            "tile_path":     tile_path,
            "magnification": tile_ann.get("magnification"),
            "x":             tile_ann.get("x"),
            "y":             tile_ann.get("y"),
            "cells":         tile_ann.get("cells", []),
        })
        found += 1
        if found == len(wanted_paths):
            break

    del tiles
    print(f"found {found}")

# Write one cell_labels.json per batch
print()
for batch_idx in range(1, NUM_BATCHES + 1):
    tiles_data = batch_tile_data[batch_idx]
    out_path = os.path.join(OUTPUT_DIR, f"batch_{batch_idx}", "cell_labels.json")
    with open(out_path, "w") as f:
        json.dump({"batch": batch_idx, "tiles": tiles_data}, f, indent=2)
    n_cells  = sum(len(t["cells"]) for t in tiles_data)
    size_mb  = os.path.getsize(out_path) / (1024 * 1024)
    print(f"  batch_{batch_idx}/cell_labels.json → {len(tiles_data)} tiles, {n_cells:,} cells ({size_mb:.1f} MB)")

## 8. Save manifest & compress

In [ ]:
# Write manifest with tile metadata including batch assignment
manifest = {
    "annotations_zip":      ANNOTATIONS_ZIP,
    "annotations_subdir":   ANNOTATIONS_SUBDIR,
    "tiles_base_uris":      TILES_BASE_URIS,
    "num_batches":          NUM_BATCHES,
    "cells_per_batch":      CELLS_PER_BATCH,
    "random_seed":          RANDOM_SEED,
    "total_tiles_sampled":  len(sampled_tiles),
    "total_cells_sampled":  cumulative_cells,
    "batches": {
        f"batch_{b}": {
            "tiles": sum(1 for t in sampled_tiles if t["batch"] == b),
            "cells": sum(t["cell_count"] for t in sampled_tiles if t["batch"] == b),
        }
        for b in range(1, NUM_BATCHES + 1)
    },
    "tiles": [
        {
            "batch":         t["batch"],
            "slide_id":      t["slide_id"],
            "tile_name":     t["tile_name"],
            "cell_count":    t["cell_count"],
            "magnification": t["magnification"],
            "x":             t["x"],
            "y":             t["y"],
            "base_uri":      slide_uri_map.get(t["slide_id"]),
            "local_path":    t.get("local_path"),
        }
        for t in sampled_tiles
    ],
}

manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest written to {manifest_path}")

In [44]:
# Done with the annotation zip
zf.close()

# Compress output into a zip archive
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as out_zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for fname in files:
            full_path = os.path.join(root, fname)
            arcname = os.path.relpath(full_path, start=".")  # preserve qa_sample/ prefix
            out_zf.write(full_path, arcname)

zip_size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
print(f"\n✅ Created {ZIP_NAME} ({zip_size_mb:.1f} MB)")
print(f"   {len(sampled_tiles)} tiles  |  {cumulative_cells:,} cells")


✅ Created qa_sample.zip (12.5 MB)
   29 tiles  |  3,696 cells


## 9. Summary

In [ ]:
print("=" * 60)
print("QA SAMPLING SUMMARY")
print("=" * 60)
print(f"Annotations zip     : {ANNOTATIONS_ZIP}")
print(f"Tiles base URIs     : {len(TILES_BASE_URIS)} URI(s)")
for u in TILES_BASE_URIS:
    print(f"  - {u}")
print(f"Slides mapped       : {len(slide_uri_map)}")
print(f"Tiles sampled       : {len(sampled_tiles)}")
print(f"Total cells         : {cumulative_cells:,}")
print(f"Slides represented  : {len(set(t['slide_id'] for t in sampled_tiles))}")
print(f"Batches             : {NUM_BATCHES}")
for b in range(1, NUM_BATCHES + 1):
    bt = [t for t in sampled_tiles if t["batch"] == b]
    print(f"  batch_{b}: {len(bt)} tiles, {sum(t['cell_count'] for t in bt):,} cells")
print(f"Output directory    : {os.path.abspath(OUTPUT_DIR)}")
print(f"Zip archive         : {os.path.abspath(ZIP_NAME)}")
print("=" * 60)
print("\nDownload the zip from the AzureML notebook file browser.")